In [ ]:
import pandas as pd

df = pd.read_pickle("../scripts/data/checkpoint_after_cleaning.pkl")

df = df.rename(columns={'UTC': 'utc'})

df.head(10)

### Analyze missing data for each feature

In [ ]:
pollutant = ['PM2.5', 'OZONE', 'NO2', 'SO2', 'CO', 'PM10']
weather_features = ['temperature_2m', 'precipitation', 'weather_code', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'relative_humidity_2m']

all_features = ['latitude', 'longitude', 'utc'] + weather_features + pollutant
existing_features = [f for f in all_features if f in df.columns]

missing_features = pd.DataFrame({
    'Feature': existing_features,
    'MissingCount': df[existing_features].isnull().sum(),
    'MissingPercent': (df[existing_features].isnull().sum() / len(df)) * 100,
})

missing_features_summary = missing_features[missing_features['MissingCount'] > 0].sort_values('MissingPercent', ascending=False)

print(missing_features_summary)

### Identify which pollutants are measured at each location

In [ ]:
pollutants = ['PM2.5', 'OZONE', 'NO2', 'SO2', 'CO', 'PM10']

locations = df.groupby(['latitude', 'longitude'])

for (lat, lon), group in locations:
    measured = []
    for pollutant in pollutants:
        if group[pollutant].notna().any():
            measured.append(pollutant)

    print(f"({lat:.4f}, {lon:.4f}) → {measured}")

###  Drop sites that do not measure label pm2.5.

In [ ]:
pm25_by_location = df.groupby(['latitude', 'longitude'])['PM2.5'].apply(lambda x: x.notna().any())

valid_locations = pm25_by_location[pm25_by_location].index

print(f"\nSites with PM2.5 sensor: {len(valid_locations)}")
for lat, lon in valid_locations:
    print(f"({lat:.4f}, {lon:.4f})")

#### keep only rows from valid locations

In [ ]:
df = df[df.set_index(['latitude', 'longitude']).index.isin(valid_locations)].reset_index(drop=True)

df.head(10)

## Check Missing Data Again

In [ ]:

missing_features = pd.DataFrame({
    'Feature': existing_features,
    'MissingCount': df[existing_features].isnull().sum(),
    'MissingPercent': (df[existing_features].isnull().sum() / len(df)) * 100,
})

missing_features_summary = missing_features[missing_features['MissingCount'] > 0].sort_values('MissingPercent', ascending=False)

print(missing_features_summary)

## Drop columns with more than % missing data

In [ ]:
from matplotlib import pyplot as plt

percantage_threshold = 55

missing_percentage = df[pollutants].isna().mean().mul(100).sort_values()

interpulate_cols = pollutants

for pol, pct in missing_percentage.items():
    if pct > percantage_threshold:
        df = df.drop(columns=[pol])
        interpulate_cols.remove(pol)

plt.figure(figsize=(8,5))
plt.bar(missing_percentage.index, missing_percentage.values)
plt.axhline(percantage_threshold, color='red', linestyle='--')
plt.ylabel('% missing'); plt.title('Missing data by pollutant')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

### Flag and interpolate missing feature data

In [ ]:

features = ['OZONE', 'PM10']

df.sort_values(['latitude', 'longitude', 'utc'])

for col in features:
    if col not in df.columns:
        continue

    df[col + '_missing_flag'] = df[col].isna().astype(int)

    df[col] = df.groupby(['latitude', 'longitude'])[col].transform(lambda x: x.interpolate(method='linear', limit=6).ffill().bfill())

    df[col] = df[col].fillna(df[col].median())


df.head(10)

### Convert UTC to datetime

In [ ]:
import numpy as np

# convert UTC to date/time
df['utc'] = pd.to_datetime(df['utc'], errors='coerce', utc=True)
# get hour of day
df['hr'] = df['utc'].dt.hour
# sine and cosine transformation for hour
df['sin_hr'] = np.sin(2 * np.pi * df['hr'] / 24)
df['cos_hr'] = np.cos(2 * np.pi * df['hr'] / 24)
# sine and cosine transformation for days
df['day'] = df['utc'].dt.dayofweek
df['sin_day'] = np.sin(2 * np.pi * df['day'] / 7)
df['cos_day'] = np.cos(2 * np.pi * df['day'] / 7)

df.head(10)

In [ ]:
# PM2.5 by hour of day (rush hour patterns)
pm25_hour = df.groupby('hr')['PM2.5'].mean()

plt.figure(figsize=(10,6))
plt.plot(pm25_hour.index, pm25_hour.values, marker='o', linewidth=2)
plt.title("Average PM2.5 Concentration by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("PM2.5")
plt.grid(alpha=0.3)
plt.xticks(range(24))
plt.show()

# PM2.5 by day of week (weekday vs weekend)
pm25_day = df.groupby('day')['PM2.5'].mean()

plt.figure(figsize=(10,6))
plt.bar(pm25_day.index, pm25_day.values)
plt.title("Average PM2.5 Concentration by Day of Week")
plt.xlabel("Day (0=Monday, 6=Sunday)")
plt.ylabel("PM2.5")
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.grid(alpha=0.3, axis='y')
plt.show()


# PM2.5 vs Temperature
plt.figure(figsize=(10,6))
plt.scatter(df['temperature_2m'], df['PM2.5'], alpha=0.1, s=1)
plt.title("PM2.5 vs Temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel("PM2.5")
plt.grid(alpha=0.3)
plt.show()

# Time series of PM2.5 (sample one site)
sample_site = df.groupby(['latitude', 'longitude']).size().idxmax()
site_data = df[(df['latitude'] == sample_site[0]) & (df['longitude'] == sample_site[1])].sort_values('utc')

plt.figure(figsize=(14,6))
plt.plot(site_data['utc'], site_data['PM2.5'], linewidth=0.5)
plt.title(f"PM2.5 Time Series for Site ({sample_site[0]:.4f}, {sample_site[1]:.4f})")
plt.xlabel("Date")
plt.ylabel("PM2.5")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# PM2.5 and Wind Speed by Hour
pm25_hour = df.groupby('hr')['PM2.5'].mean()
wind_hour = df.groupby('hr')['wind_speed_10m'].mean()

plt.figure(figsize=(10,6))
plt.plot(pm25_hour.index, pm25_hour.values, label='PM2.5', marker='o', linewidth=2)
plt.plot(wind_hour.index, wind_hour.values, label='Wind Speed (10m)', marker='s', linewidth=2)
plt.title("PM2.5 Concentration vs Wind Speed by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Value")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
df = df.dropna(subset=['PM2.5']).reset_index(drop=True)

df.tail(10)

### Analyze Feature performance after Pre-processing

### Analyze Feature Pair Performance